In [25]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm.notebook import tqdm

In [26]:
torch.manual_seed(11037)

In [27]:
class ProjectRedMLP_SSJG2(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.hidden = nn.Sequential(    nn.Linear(5, 64),
                                        nn.SiLU(),
                                        nn.Linear(64, 32),
                                        nn.SiLU(),        )

        self.center = nn.Sequential(    nn.Linear(32, 64),
                                        nn.SiLU(),
                                        nn.Linear(64, 16),
                                        nn.SiLU(),
                                        nn.Linear(16, 1))


        self.width =  nn.Sequential(    nn.Linear(32, 64),
                                        nn.SiLU(),
                                        nn.Linear(64, 16),
                                        nn.SiLU(),
                                        nn.Linear(16, 1))
        
    def forward(self, x):

        x = self.hidden(x)
        center = self.center(x)
        width = self.width(x) 

        return torch.cat((center, width), dim=1)

In [28]:
model = ProjectRedMLP_SSJG2()
print(model)

ProjectRedMLP_SSJG2(
  (hidden): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): SiLU()
  )
  (center): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
    (3): SiLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
  (width): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): SiLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
    (3): SiLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [29]:
df = pd.read_csv("project_red_center_width.csv")
df.head()

,user_id,last_problem_id,user_rating,last_problem_rating,mean_solve_time_minutes,time_std_minutes,problems_solved,center,width,lower_bound,upper_bound,synthetic
0,SYN_000000,2109:C3,2263.0,2600,31.77,4.34,64,2417.25,298.93,2267.78,2566.72,True
1,SYN_000001,606:B,1414.0,1600,31.28,9.94,84,1555.10,568.89,1270.65,1839.54,True
2,SYN_000002,913:D,1833.0,1800,29.39,11.96,79,1844.16,426.22,1631.05,2057.27,True
3,SYN_000003,1419:D2,1290.0,1500,41.23,17.98,142,1347.95,588.71,1053.59,1642.31,True
4,SYN_000004,1428:H,3385.0,3500,31.50,17.31,306,3396.72,226.51,3283.46,3509.97,True


In [30]:
features = ["user_rating", "last_problem_rating", "mean_solve_time_minutes", "time_std_minutes", "problems_solved"]

targets = ["center", "width"]

X = df[features]
Y = df[targets]

In [31]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=11037)

In [32]:
scaler_x = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test =  scaler_x.fit_transform(X_test)

In [33]:
joblib.dump(scaler_x, "scaler_x_v2_3.pkl")

['scaler_x_v2_3.pkl']

In [34]:
scaler_y = StandardScaler()

Y_train = scaler_y.fit_transform(Y_train)
Y_test =  scaler_y.fit_transform(Y_test)

In [35]:
X_train.dtype

dtype('float64')

In [36]:
Y_train.dtype

dtype('float64')

In [37]:
joblib.dump(scaler_y, "scaler_y_v2_3.pkl")

['scaler_y_v2_3.pkl']

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:",device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

Using: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [39]:
model = model.to(device=device)

In [40]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [41]:
train_df = TensorDataset(X_train_tensor, Y_train_tensor)
test_df = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_df, batch_size=64, shuffle=True)
test_loader = DataLoader(test_df, batch_size=64, shuffle=False)

In [42]:
criterion = nn.MSELoss()

optim = torch.optim.Adam(model.parameters(), lr=0.001)

In [43]:
epochs = 100

train_losses = []
test_losses = []

progress = tqdm(range(epochs), desc="Training Gama", unit="epochs")

for epoch in progress:
    model.train()
    total_train_loss = 0

    for X_bch, Y_bch in train_loader:
        X_bch = X_bch.to(device)
        Y_bch = Y_bch.to(device)

        optim.zero_grad()

        preds = model(X_bch)

        loss = criterion(preds, Y_bch)

        loss.backward()

        optim.step()

        total_train_loss += loss.item() * X_bch.size(0)

    avg_train_loss = total_train_loss / len(train_df)
    train_losses.append(avg_train_loss)


    model.eval()
    total_test_loss = 0

    with torch.no_grad():
        for X_bch, Y_bch in test_loader:

            X_bch = X_bch.to(device)
            Y_bch = Y_bch.to(device)

            preds = model(X_bch)

            loss = criterion(preds, Y_bch)

            total_test_loss += loss.item() * X_bch.size(0)
    avg_test_loss = total_test_loss/len(test_df)
    test_losses.append(avg_test_loss)

    progress.set_postfix({"Train loss":f"{avg_train_loss}", "Test loss":f"{avg_test_loss}"})

Training Gama:   0%|          | 0/100 [00:00<?, ?epochs/s]

In [44]:
train_losses[:10]

[0.2857052908539772,
 0.04810253804922104,
 0.04476209230720997,
 0.04266391327977181,
 0.04155525471270084,
 0.04203484834730625,
 0.04132450406253338,
 0.041653539150953295,
 0.041570885077118876,
 0.04120901633799076]

In [45]:
train_losses[90:]

[0.04105548898875713,
 0.041078187853097914,
 0.040727905020117756,
 0.040812208235263825,
 0.04070669767260551,
 0.041150080606341365,
 0.04068589018285274,
 0.04097832930088043,
 0.040916956290602685,
 0.04064653649926186]

In [46]:
model.eval()

with torch.no_grad():
    prd = model(X_test_tensor.to(device)).cpu()

    center_MeanSquareError = ((prd[:,0] - Y_test_tensor[:,0]) ** 2).mean()
    width_MeanSquareError = ((prd[:,1] - Y_test_tensor[:,1]) ** 2).mean()

print("Center MSE:", center_MeanSquareError.item())
print("Width MSE:", width_MeanSquareError.item())

Center MSE: 0.008031599223613739
Width MSE: 0.07600461691617966


In [47]:
torch.save(model.state_dict(), "gama_v2_3.pth")

In [48]:
dummy = torch.randn(1, 5, device=device)
torch.onnx.export(model, dummy, "gama_v2_3.onnx", input_names=["input"], output_names=["c/w"], opset_version=18, dynamo=True)

[torch.onnx] Obtain model graph for `ProjectRedMLP_SSJG2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ProjectRedMLP_SSJG2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/nix/store/33lb265capmbnrgmp7gghfgfx58s60lq-python3-3.13.15-env/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,5]>
            ),
            outputs=(
                %"c/w"<FLOAT,[1,2]>
            ),
            initializers=(
                %"hidden.0.weight"<FLOAT,[64,5]>{TorchTensor(...)},
                %"hidden.0.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"hidden.2.weight"<FLOAT,[32,64]>{TorchTensor(...)},
                %"hidden.2.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"center.0.weight"<FLOAT,[64,32]>{TorchTensor(...)},
                %"center.0.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"center.2.weight"<FLOAT,[16,64]>{TorchTensor(...)},
                %"center.2.bias"<FLOAT,[16]>{TorchTensor(...)},
           